# MOSTA Telencephalon velocity (scVelo stream + grid)

This notebook isolates the telencephalon zoom plotting workflow from the original MOSTA velocity analysis and adds `scv.pl.velocity_embedding_grid` for direct stream-versus-grid comparison.

The workflow is adapted to the current downstream repository and uses only in-repo MOSTA assets and outputs.


In [ ]:
# Imports
# Requires the `cb_pipeline` environment with torch, scanpy, scvelo, anndata, and matplotlib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# Repo root + sys.path
import sys
from pathlib import Path


def _find_downstream_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'downstream_helpers').is_dir() and (p / 'vendor').is_dir():
            return p
    return start


DOWNSTREAM_ROOT = _find_downstream_root(Path.cwd().resolve())
CYTOBRIDGE_REPO = (DOWNSTREAM_ROOT.parent / 'cytobridge-spatial').resolve()
VENDOR_ROOT = (DOWNSTREAM_ROOT / 'vendor').resolve()
for path in (DOWNSTREAM_ROOT, VENDOR_ROOT, CYTOBRIDGE_REPO):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print('Downstream root:', DOWNSTREAM_ROOT)
print('CytoBridge repo:', CYTOBRIDGE_REPO)

try:
    import torch
    import anndata as ad
    import scanpy as sc
    import scvelo as scv
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        'Missing dependencies: run this notebook in the cb_pipeline environment with torch, scanpy, scvelo, and anndata installed.'
    ) from e

from downstream_helpers import load_mosta_context

# Optional: keep scVelo figure defaults close to the manuscript style
scv.settings.set_figure_params('scvelo', dpi=120)


## Parameters

- Adjust `DENSITY_STREAM` and `DENSITY_GRID` to control streamline and grid density.
- To plot only one velocity component, set `PLOT_COMPONENTS` to values such as `['full']` or `['intrinsic']`.


In [ ]:
# Parameters
# This notebook uses the MOSTA data and model assets bundled with the downstream repo

EXP_NAME = 'mosta_assets_model'
RESULT_DIR_NAME = 'mosta_telencephalon_velocity_stream_grid_notebook'

# Telencephalon zoom window from the original notebook
ZOOM_REGION = [-1.3, -0.5, 3.3, 4.2]  # [x_min, x_max, y_min, y_max]

# E15.5 corresponds to samples=3.0
TIMEPOINT_VALUE = 3.0
TIMEPOINT_LABEL = 'E15.5'

# Restrict to Brain cells to match the original notebook; set to None to keep all cells
FOCUS_ANNOTATION = 'Brain'  # or None

# Stream / grid density parameters
DENSITY_STREAM = 1.0
DENSITY_GRID = 1.0

# Velocity components to plot: 'intrinsic' / 'interaction' / 'full'
PLOT_COMPONENTS = ['full']

# Additional plotting parameters
FIGSIZE = (7, 5)
REMOVE_OUTLIERS = True
FLIP_Y = False
FLIP_X = False
MODE = 'default'  # 'default' or 'black'

# Number of neighbors used for the gene-space projection
GENE_N_NEIGHBORS = 30

# Interaction-term parameters kept consistent with the original notebook
INTERACTION_M = 1024
INTERACTION_THRESHOLD = 1000


In [ ]:
# Load models + data from current downstream repo assets
from evaluation.arista_code import arista_helpers as helpers
from evaluation.arista_code.mosta_ported import velocity as velmod

context = load_mosta_context()
f_net = context.runtime.f_net
score_net = context.runtime.score_net
device = context.device
df = context.df.copy()
df['samples'] = df['samples'].astype(float)
print('Loaded MOSTA runtime from assets | device:', device)

# Subset the selected timepoint
sel = df['samples'] == float(TIMEPOINT_VALUE)
df_t = df.loc[sel].copy()
print('Rows at timepoint:', len(df_t))

# Restrict to Brain cells (optional)
if FOCUS_ANNOTATION is not None and 'Annotation' in df_t.columns:
    df_t = df_t[df_t['Annotation'].astype(str) == str(FOCUS_ANNOTATION)].copy()
    print('Rows after Annotation filter:', len(df_t))

if df_t.empty:
    raise ValueError('df_t is empty: check TIMEPOINT_VALUE / FOCUS_ANNOTATION')

# Model input: x1..x52
feature_cols = [f'x{i}' for i in range(1, int(context.dim) + 1)]
all_data = df_t[feature_cols].to_numpy(dtype=np.float32)
coords = all_data[:, :2]
X_expression = all_data[:, 2:]

# Compute velocity components once (drift, interaction, score, full)
vel = helpers.compute_velocity_components(
    all_data,
    float(TIMEPOINT_VALUE),
    f_net,
    score_net,
    interaction_m=INTERACTION_M,
    interaction_threshold=INTERACTION_THRESHOLD,
    device=device,
)


In [ ]:
# Telencephalon palette (matches mosta_velocity.ipynb / mosta_velocity_local.py)
TELENCEPHALON_PALETTE = {
    'Apical Progenitors (RG)': '#1f77b4',
    'Basal Progenitors (IP)': '#aec7e8',
    'Immature Neurons': '#2ca02c',
    'Excitatory Neurons': '#ffbb78',
    'Inhibitory Neurons': '#9467bd',
    'Cajal-Retzius Cells': '#e377c2',
    'Glioblasts': '#8c564b',
    'Choroid Plexus': '#7f7f7f',
    'Other': '#d9d9d9',
}

# Ensure the telencephalon column exists; otherwise derive it from celltype labels
if 'telencephalon' not in df_t.columns and 'celltype' in df_t.columns:
    mapping_dict = {
        'Forebrain radial glia': 'Apical Progenitors (RG)',
        'Cortical intermediate progenitor': 'Basal Progenitors (IP)',
        'Cortical glutamatergic neuroblast': 'Immature Neurons',
        'Forebrain glutamatergic neuroblast': 'Immature Neurons',
        'Forebrain neuroblast': 'Immature Neurons',
        'Cortical glutamatergic neuron': 'Excitatory Neurons',
        'Cortical or hippocampal glutamatergic neuron': 'Excitatory Neurons',
        'Forebrain GABAergic neuron': 'Inhibitory Neurons',
        'Forebrain GABAergic neuroblast': 'Inhibitory Neurons',
        'Cajal-Retzius cell': 'Cajal-Retzius Cells',
        'Hindbrain glioblast': 'Glioblasts',
        'Mixed region glioblast': 'Glioblasts',
        'Choroid plexus': 'Choroid Plexus',
        'Choroid plexus progenitor': 'Choroid Plexus',
    }
    df_t['telencephalon'] = df_t['celltype'].astype(str).map(mapping_dict).fillna('Other')

allowed = set(TELENCEPHALON_PALETTE.keys())
# Collapse unknown categories into Other
if 'telencephalon' in df_t.columns:
    df_t['telencephalon'] = df_t['telencephalon'].astype(str)
    df_t.loc[~df_t['telencephalon'].isin(allowed), 'telencephalon'] = 'Other'


In [ ]:
# Helpers: gene-space projection + grid plot wrapper

def project_gene_velocity_to_2d(V_high_dim: np.ndarray, *, n_neighbors: int) -> np.ndarray:
    tmp_ad = ad.AnnData(X=X_expression)
    tmp_ad.layers['Ms'] = X_expression.copy()
    tmp_ad.obsm['X_spatial'] = coords.copy()

    sc.pp.neighbors(tmp_ad, n_neighbors=int(n_neighbors), use_rep='X')
    tmp_ad.layers['velocity'] = V_high_dim.copy()

    scv.tl.velocity_graph(tmp_ad, vkey='velocity', xkey='Ms', n_jobs=1)
    scv.tl.velocity_embedding(tmp_ad, basis='spatial', vkey='velocity')
    return tmp_ad.obsm['velocity_spatial'].copy()


def plot_single_velocity_field_grid(
    adata,
    velocity_key: str,
    density: float,
    figsize,
    flip_y: bool,
    flip_x: bool,
    title: str,
    color_key: str,
    mode: str = 'default',
    remove_outliers: bool = True,
    timepoint_str=None,
    plot_region=None,
    palette=None,
    **kwargs,
):
    # This follows velmod.plot_single_velocity_field but switches from stream to grid rendering
    alpha_val = kwargs.get('alpha', 0.25)

    if mode == 'black':
        plt.style.use('dark_background')
        background_color, text_color = 'black', 'white'
    else:
        plt.style.use('default')
        background_color, text_color = 'white', 'black'

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(background_color)
    ax.set_facecolor(background_color)

    adata_plot = adata.copy()
    if remove_outliers:
        y = adata_plot.obsm['X_spatial'][:, 1]
        q1, q3 = np.percentile(y, [25, 75])
        iqr = q3 - q1
        mask = (y >= q1 - 1.5 * iqr) & (y <= q3 + 1.5 * iqr)
        adata_plot = adata_plot[mask].copy()

    if plot_region is not None:
        x_min, x_max, y_min, y_max = plot_region
        X = adata_plot.obsm['X_spatial']
        adata_before_zoom = adata_plot
        mask = np.ones(len(X), dtype=bool)
        if x_min is not None:
            mask &= X[:, 0] > x_min
        if x_max is not None:
            mask &= X[:, 0] < x_max
        if y_min is not None:
            mask &= X[:, 1] > y_min
        if y_max is not None:
            mask &= X[:, 1] < y_max
        adata_plot = adata_plot[mask].copy()
        if int(adata_plot.n_obs) == 0:
            print('  Zoom subset is empty; falling back to the unzoomed view.')
            adata_plot = adata_before_zoom
        else:
            print(f'  Zoom subset: {len(adata_plot)} cells remaining.')

    point_size = 50 if plot_region is None else 60
    grid_n_neighbors = max(1, min(30, int(adata_plot.n_obs) - 1)) if int(adata_plot.n_obs) > 1 else 1
    palette_arg = velmod.palette_to_sequence(adata_plot, color_key, palette)

    scv.pl.velocity_embedding_grid(
        adata_plot,
        basis='spatial',
        vkey=velocity_key,
        color=color_key,
        palette=palette_arg,
        ax=ax,
        show=False,
        density=density,
        n_neighbors=grid_n_neighbors,
        arrow_size=1.4,
        linewidth=1.0,
        alpha=alpha_val,
        size=point_size,
        legend_loc='right margin',
        title='',
        frameon=False,
    )

    if flip_y:
        ax.invert_yaxis()
    if flip_x:
        ax.invert_xaxis()

    if plot_region is not None:
        x_min, x_max, y_min, y_max = plot_region
        cur_xlim = ax.get_xlim()
        cur_ylim = ax.get_ylim()
        ax.set_xlim(x_min if x_min is not None else cur_xlim[0], x_max if x_max is not None else cur_xlim[1])
        target_ymin = y_min if y_min is not None else (cur_ylim[0] if not flip_y else cur_ylim[1])
        target_ymax = y_max if y_max is not None else (cur_ylim[1] if not flip_y else cur_ylim[0])
        if flip_y:
            ax.set_ylim(target_ymax, target_ymin)
        else:
            ax.set_ylim(target_ymin, target_ymax)

    full_title = f"{title} - {timepoint_str}" if timepoint_str else title
    ax.set_title(full_title, fontsize=18, fontweight='bold', color=text_color, pad=14)
    for spine in ax.spines.values():
        spine.set_color(text_color)
    ax.tick_params(colors=text_color, labelsize=10)
    ax.xaxis.label.set_color(text_color)
    ax.yaxis.label.set_color(text_color)

    return fig, ax


## Plotting: physical / gene space (stream vs grid)

By default the notebook plots only `full`. To inspect `intrinsic` or `interaction`, update `PLOT_COMPONENTS`.


In [ ]:
# Build an AnnData for plotting (same shape as original notebook)

OUT_DIR = DOWNSTREAM_ROOT / 'results' / RESULT_DIR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save both PNG (raster) and SVG (vector)
SAVE_FORMATS = ('png', 'svg')


def _clean_filename_part(x) -> str:
    return (
        str(x)
        .strip()
        .replace(' ', '_')
        .replace('/', '-')
        .replace('\\', '-')
        .replace(':', '-')
    )


def _out_path(space: str, comp: str, kind: str, ext: str):
    exp = _clean_filename_part(EXP_NAME)
    tp = _clean_filename_part(TIMEPOINT_LABEL)
    return OUT_DIR / f'{exp}_{tp}_{space}_{comp}_{kind}.{ext}'


def _save_fig(fig, space: str, comp: str, kind: str):
    for ext in SAVE_FORMATS:
        out = _out_path(space, comp, kind, ext)
        if ext.lower() == 'svg':
            fig.savefig(out, format='svg', bbox_inches='tight')
        else:
            fig.savefig(out, dpi=300, bbox_inches='tight')
        print('Saved:', out.resolve())


def build_telencephalon_adata(space, components):
    if space not in {'physical', 'gene'}:
        raise ValueError(space)

    V_2d = {}
    for comp in components:
        if comp == 'intrinsic':
            V_high = vel['drift']
        elif comp == 'interaction':
            V_high = vel['interaction']
        elif comp == 'full':
            V_high = vel['full']
        else:
            raise ValueError(comp)

        if space == 'physical':
            V_2d[comp] = V_high[:, :2]
        else:
            V_2d[comp] = project_gene_velocity_to_2d(V_high[:, 2:], n_neighbors=GENE_N_NEIGHBORS)

    adata_tel = ad.AnnData(X=all_data)
    adata_tel.obsm['X_spatial'] = coords

    for comp, v in V_2d.items():
        adata_tel.obsm[f'velocity_{comp}_spatial'] = v

    adata_tel.obs['telencephalon'] = df_t['telencephalon'].astype(str).to_numpy()
    adata_tel.obs['telencephalon'] = pd.Categorical(adata_tel.obs['telencephalon'])

    pal = velmod.ensure_valid_palette(adata_tel, 'telencephalon', TELENCEPHALON_PALETTE)
    return adata_tel, pal


def plot_space(space, components):
    adata_tel, pal = build_telencephalon_adata(space, components)

    space_title = 'Physical Space' if space == 'physical' else 'Gene Space'

    for comp in components:
        vkey = f'velocity_{comp}'
        title = f'{space_title} - {comp.capitalize()} Velocity'

        fig, ax = velmod.plot_single_velocity_field(
            adata_tel,
            velocity_key=vkey,
            density=DENSITY_STREAM,
            figsize=FIGSIZE,
            flip_y=FLIP_Y,
            flip_x=FLIP_X,
            title=f'{title} (stream)',
            color_key='telencephalon',
            mode=MODE,
            remove_outliers=REMOVE_OUTLIERS,
            timepoint_str=TIMEPOINT_LABEL,
            plot_region=ZOOM_REGION,
            palette=pal,
        )
        _save_fig(fig, space, comp, 'stream')
        plt.show()
        plt.close(fig)

        fig, ax = plot_single_velocity_field_grid(
            adata_tel,
            velocity_key=vkey,
            density=DENSITY_GRID,
            figsize=FIGSIZE,
            flip_y=FLIP_Y,
            flip_x=FLIP_X,
            title=f'{title} (grid)',
            color_key='telencephalon',
            mode=MODE,
            remove_outliers=REMOVE_OUTLIERS,
            timepoint_str=TIMEPOINT_LABEL,
            plot_region=ZOOM_REGION,
            palette=pal,
        )
        _save_fig(fig, space, comp, 'grid')
        plt.show()
        plt.close(fig)


## Quick plot: gene-space interaction (density = 1)

This cell temporarily sets `DENSITY_STREAM` and `DENSITY_GRID` to 1, renders `gene + interaction`, and then restores the previous density values.


In [ ]:
# Gene-space interaction with density=1
_old_stream, _old_grid = DENSITY_STREAM, DENSITY_GRID
DENSITY_STREAM = 1.0
DENSITY_GRID = 1.0

plot_space('gene', ['interaction'])

DENSITY_STREAM, DENSITY_GRID = _old_stream, _old_grid


In [ ]:
# Physical-space interaction with density=1
_old_stream, _old_grid = DENSITY_STREAM, DENSITY_GRID
DENSITY_STREAM = 1.0
DENSITY_GRID = 1.0

plot_space('physical', ['interaction'])

DENSITY_STREAM, DENSITY_GRID = _old_stream, _old_grid
